In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/Chattogram_Landmark_Dataset"

FINAL_DIR = os.path.join(
    PROJECT_ROOT,
    "05_final_dataset"
)

SOURCE_DIR = os.path.join(
    FINAL_DIR,
    "images"
)

SPLIT_DIR = os.path.join(
    PROJECT_ROOT,
    "06_train_val_test"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "07_models"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "08_results"
)

for folder in [
    SPLIT_DIR,
    MODEL_DIR,
    RESULT_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Project directories ready.")

print("\nSource dataset:")
print(SOURCE_DIR)

print("\nSplit dataset:")
print(SPLIT_DIR)

print("\nModels:")
print(MODEL_DIR)

print("\nResults:")
print(RESULT_DIR)

Project directories ready.

Source dataset:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/05_final_dataset/images

Split dataset:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test

Models:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/07_models

Results:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/08_results


In [ ]:
import os
import pandas as pd

IMAGE_DIR = SOURCE_DIR

if not os.path.exists(IMAGE_DIR):
    raise FileNotFoundError(
        f"Final image directory not found:\n{IMAGE_DIR}"
    )

class_info = []

for district in sorted(os.listdir(IMAGE_DIR)):

    district_path = os.path.join(
        IMAGE_DIR,
        district
    )

    if not os.path.isdir(district_path):
        continue

    for landmark in sorted(os.listdir(district_path)):

        landmark_path = os.path.join(
            district_path,
            landmark
        )

        if not os.path.isdir(landmark_path):
            continue

        images = [
            f for f in os.listdir(landmark_path)
            if os.path.isfile(
                os.path.join(landmark_path, f)
            )
        ]

        class_info.append({
            "district": district,
            "landmark": landmark,
            "image_count": len(images)
        })

class_df = pd.DataFrame(class_info)

print("Total classes:", len(class_df))
print("Total images:", class_df["image_count"].sum())

print("\nMinimum images/class:",
      class_df["image_count"].min())

print("Maximum images/class:",
      class_df["image_count"].max())

print("\nClasses below 30:",
      (class_df["image_count"] < 30).sum())

print("Classes above 40:",
      (class_df["image_count"] > 40).sum())

Total classes: 58
Total images: 2137

Minimum images/class: 31
Maximum images/class: 40

Classes below 30: 0
Classes above 40: 0


In [ ]:
# ============================================================
# LOAD EXISTING SPLIT MANIFEST
# ============================================================

import os
import pandas as pd

SPLIT_DIR = "/content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test"

SPLIT_CSV = os.path.join(
    SPLIT_DIR,
    "split_manifest.csv"
)

print("Checking:")
print(SPLIT_CSV)

if not os.path.exists(SPLIT_CSV):
    raise FileNotFoundError(
        f"split_manifest.csv not found:\n{SPLIT_CSV}"
    )

split_df = pd.read_csv(
    SPLIT_CSV
)

print("\n✓ Split manifest loaded successfully.")

print("Total records:", len(split_df))

print("\nColumns:")
print(split_df.columns.tolist())

print("\nFirst 5 rows:")
display(split_df.head())

Checking:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/split_manifest.csv

✓ Split manifest loaded successfully.
Total records: 2137

Columns:
['district', 'landmark', 'filename', 'split', 'source_path', 'split_path']

First 5 rows:


,district,landmark,filename,split,source_path,split_path
0,Bandarban,Boga_Lake,0019_File_Boga_Lake_22.jpg.jpg,train,/content/drive/MyDrive/Chattogram_Landmark_Dat...,/content/drive/MyDrive/Chattogram_Landmark_Dat...
1,Bandarban,Boga_Lake,0010_File_Boga_Lake_13.jpg.jpg,train,/content/drive/MyDrive/Chattogram_Landmark_Dat...,/content/drive/MyDrive/Chattogram_Landmark_Dat...
2,Bandarban,Boga_Lake,0023_File_Boga_Lake_26.jpg.jpg,train,/content/drive/MyDrive/Chattogram_Landmark_Dat...,/content/drive/MyDrive/Chattogram_Landmark_Dat...
3,Bandarban,Boga_Lake,0092_File_Young_Adivasi_Indigenous_Man_-_En_ro...,train,/content/drive/MyDrive/Chattogram_Landmark_Dat...,/content/drive/MyDrive/Chattogram_Landmark_Dat...
4,Bandarban,Boga_Lake,0065_File_The_Wooden_Cabin_6822597340_.jpg.jpg,train,/content/drive/MyDrive/Chattogram_Landmark_Dat...,/content/drive/MyDrive/Chattogram_Landmark_Dat...


In [ ]:
verification = (
    split_df
    .groupby(["district", "landmark", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(verification)

split,district,landmark,test,train,val
0,Bandarban,Boga_Lake,6,28,6
1,Bandarban,Chimbuk_Hill,6,24,5
2,Bandarban,Golden_Temple_Bandarban,6,24,5
3,Bandarban,Keokradong,6,27,6
4,Bandarban,Nilachal,6,28,6
5,Bandarban,Nilgiri,5,26,6
6,Brahmanbaria,Anderson_Canal,5,26,6
7,Brahmanbaria,Arifail_Mosque,4,22,5
8,Brahmanbaria,Haripur_Barendra_Bari,6,25,5
9,Brahmanbaria,Titas_River,6,25,5


In [ ]:
print("========================================")
print("SPLIT VERIFICATION")
print("========================================")

print("Train:",
      (split_df["split"] == "train").sum())

print("Validation:",
      (split_df["split"] == "val").sum())

print("Test:",
      (split_df["split"] == "test").sum())

print("\nClasses:")
print(split_df["landmark"].nunique())

# Check every class has all three splits
split_presence = (
    split_df
    .groupby(["district", "landmark"])["split"]
    .nunique()
)

print(
    "\nClasses missing a split:",
    (split_presence < 3).sum()
)

if (split_presence == 3).all():
    print("\n✓ Every class exists in train, validation and test.")
else:
    print("\n⚠ Some classes are missing a split.")

SPLIT VERIFICATION
Train: 1488
Validation: 321
Test: 328

Classes:
58

Classes missing a split: 0

✓ Every class exists in train, validation and test.


In [ ]:
duplicate_split_check = (
    split_df
    .groupby(["district", "landmark", "filename"])
    .size()
)

duplicates = duplicate_split_check[
    duplicate_split_check > 1
]

print(
    "Images appearing in multiple splits:",
    len(duplicates)
)

if len(duplicates) == 0:
    print("✓ No image appears in multiple splits.")
else:
    print("⚠ Duplicate split assignments detected.")

Images appearing in multiple splits: 0
✓ No image appears in multiple splits.


In [ ]:
classes = sorted(
    split_df["landmark"].unique()
)

label_to_index = {
    label: index
    for index, label in enumerate(classes)
}

index_to_label = {
    index: label
    for label, index in label_to_index.items()
}

labels_df = pd.DataFrame({
    "index": range(len(classes)),
    "landmark": classes
})

LABEL_CSV = os.path.join(
    SPLIT_DIR,
    "class_labels.csv"
)

labels_df.to_csv(
    LABEL_CSV,
    index=False
)

print("Number of classes:", len(classes))
print("\nClass mapping:")
display(labels_df)

Number of classes: 58

Class mapping:


,index,landmark
0,0,Adinath_Temple
1,1,Alutila_Cave
2,2,Anderkilla_Shahi_Jame_Mosque
3,3,Anderson_Canal
4,4,Arifail_Mosque
5,5,Bajra_Shahi_Mosque
6,6,Banshbaria_Beach
7,7,Batali_Hill
8,8,Bayazid_Bostami_Shrine
9,9,Boga_Lake


In [ ]:
MODEL_NAMES = [
    "ResNet50",
    "EfficientNetB0",
    "MobileNetV3Large"
]

for model_name in MODEL_NAMES:

    model_path = os.path.join(
        MODEL_DIR,
        model_name
    )

    os.makedirs(
        model_path,
        exist_ok=True
    )

print("Model folders created:")

for model_name in MODEL_NAMES:
    print(
        os.path.join(
            MODEL_DIR,
            model_name
        )
    )

Model folders created:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/07_models/ResNet50
/content/drive/MyDrive/Chattogram_Landmark_Dataset/07_models/EfficientNetB0
/content/drive/MyDrive/Chattogram_Landmark_Dataset/07_models/MobileNetV3Large


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

PyTorch version: 2.11.0+cpu
CUDA available: False


In [ ]:
import os
import json
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/Chattogram_Landmark_Dataset"

SPLIT_DIR = os.path.join(
    PROJECT_ROOT,
    "06_train_val_test"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "07_models",
    "ResNet50"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "08_results",
    "ResNet50"
)

TRAIN_DIR = os.path.join(
    SPLIT_DIR,
    "train"
)

VAL_DIR = os.path.join(
    SPLIT_DIR,
    "val"
)

TEST_DIR = os.path.join(
    SPLIT_DIR,
    "test"
)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Test:", TEST_DIR)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULT_DIR)

Train: /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/train
Validation: /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/val
Test: /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/test
Model directory: /content/drive/MyDrive/Chattogram_Landmark_Dataset/07_models/ResNet50
Results directory: /content/drive/MyDrive/Chattogram_Landmark_Dataset/08_results/ResNet50


In [ ]:
import os

DATASET_ROOT = "/content/drive/MyDrive/Chattogram_Landmark_Dataset"
SPLIT_DIR = os.path.join(DATASET_ROOT, "06_train_val_test")

print("Dataset root exists:", os.path.exists(DATASET_ROOT))
print("Split directory exists:", os.path.exists(SPLIT_DIR))

print("\n========== SPLIT DIRECTORY ==========")

if os.path.exists(SPLIT_DIR):
    for item in sorted(os.listdir(SPLIT_DIR)):
        full_path = os.path.join(SPLIT_DIR, item)

        if os.path.isdir(full_path):
            print("📁", item)
        else:
            print("📄", item)

print("\n========== EXACT PATH CHECK ==========")

for name in ["train", "val", "test"]:
    path = os.path.join(SPLIT_DIR, name)

    print(
        f"{name:5} -> {path} -> "
        f"{'EXISTS ✓' if os.path.isdir(path) else 'MISSING ❌'}"
    )

Dataset root exists: True
Split directory exists: True

========== SPLIT DIRECTORY ==========
📄 class_labels.csv
📄 split_manifest.csv
📁 test
📁 train
📁 val

========== EXACT PATH CHECK ==========
train -> /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/train -> EXISTS ✓
val   -> /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/val -> EXISTS ✓
test  -> /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/test -> EXISTS ✓


In [ ]:
import os

DATASET_ROOT = "/content/drive/MyDrive/Chattogram_Landmark_Dataset"

print("Dataset root:", DATASET_ROOT)
print("Exists:", os.path.exists(DATASET_ROOT))

print("\nDataset folders/files:")

for item in sorted(os.listdir(DATASET_ROOT)):
    full_path = os.path.join(DATASET_ROOT, item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

Dataset root: /content/drive/MyDrive/Chattogram_Landmark_Dataset
Exists: True

Dataset folders/files:
📁 01_inventory
📁 02_availability
📁 03_candidates
📁 04_review
📁 05_final_dataset
📁 06_train_val_test
📁 07_models
📁 08_results
📁 09_cache


In [ ]:
SPLIT_DIR = os.path.join(
    DATASET_ROOT,
    "06_train_val_test"
)

print("SPLIT_DIR:")
print(SPLIT_DIR)

print("\nExists:", os.path.isdir(SPLIT_DIR))

if os.path.isdir(SPLIT_DIR):
    print("\nContents:")
    for item in sorted(os.listdir(SPLIT_DIR)):
        full_path = os.path.join(SPLIT_DIR, item)

        if os.path.isdir(full_path):
            print("📁", item)
        else:
            print("📄", item)

SPLIT_DIR:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test

Exists: True

Contents:
📄 class_labels.csv
📄 split_manifest.csv
📁 test
📁 train
📁 val


In [ ]:
for name in ["train", "val", "test"]:

    path = os.path.join(SPLIT_DIR, name)

    print(
        name,
        "->",
        "✓ EXISTS" if os.path.isdir(path) else "❌ MISSING",
        "\n ",
        path
    )

train -> ✓ EXISTS 
  /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/train
val -> ✓ EXISTS 
  /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/val
test -> ✓ EXISTS 
  /content/drive/MyDrive/Chattogram_Landmark_Dataset/06_train_val_test/test
